# Spam filter: benchmark five approaches

Compares Naive Bayes, a linear SVM, TF-IDF + logistic regression, the fine-tuned DistilBERT and a free
open-weight LLM (Qwen2.5-3B-Instruct, zero-shot, running on this GPU) on the same Enron test emails and the
same 2,000 modern phishing-dataset emails. No API key needed.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.* Then run the cells top to bottom.
Step 2 restarts the runtime once on purpose. Total: about 10 minutes if your model is saved in Google
Drive, about 40 minutes if it has to be retrained. The first run also downloads the LLM (~6 GB, 1–2 min).

## 1. Get the code

In [ ]:
%cd /content
!rm -rf Email-Filtering-Using-DistilBERT
!git clone -q -b feature/hybrid-api https://github.com/namita-tj/Email-Filtering-Using-DistilBERT.git
%cd /content/Email-Filtering-Using-DistilBERT
!git log --oneline -1
import os
assert os.path.exists("app/local_llm.py"), "New code missing: push the latest changes to GitHub first."

## 2. Install (the runtime restarts by itself — then continue with step 3)

In [ ]:
%cd /content/Email-Filtering-Using-DistilBERT
!pip install -q "tensorflow==2.20.0" "tf_keras==2.20.1" "transformers==4.49.0" "datasets==3.3.2" "scikit-learn==1.6.1" "fastapi==0.115.6" "uvicorn==0.34.0" "pydantic==2.10.6" "openai==1.59.7" "boto3==1.37.9"
import os
os.kill(os.getpid(), 9)

## 3. Setup check

In [ ]:
%cd /content/Email-Filtering-Using-DistilBERT
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import torch
print("TensorFlow", tf.__version__, "| PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then start again from step 1."

## 4. Get the trained model

Looks for the model you saved to Google Drive in the training notebook (step 9). If it is not there,
set `RETRAIN = True` and run the cell again: training takes about 27 minutes on a T4.

In [ ]:
import os, shutil
RETRAIN = False
DRIVE_MODEL = "/content/drive/MyDrive/spam_filter/spam_detection_model"

if not os.path.exists("spam_detection_model/tf_model.h5"):
    if RETRAIN:
        os.environ.update({"MAX_LENGTH": "256", "EPOCHS": "2", "TRAIN_LIMIT": "0"})
        !python -m training.train
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.exists(DRIVE_MODEL):
            shutil.copytree(DRIVE_MODEL, "spam_detection_model", dirs_exist_ok=True)
            print("Model copied from Google Drive.")
        else:
            print("No model in Google Drive. Set RETRAIN = True above and run this cell again.")
print("Model ready:", os.path.exists("spam_detection_model/tf_model.h5"))

## 5. Run the benchmark (≈ 10–15 min)

In [ ]:
import os
os.environ.update({"TF_USE_LEGACY_KERAS": "1", "TF_FORCE_GPU_ALLOW_GROWTH": "true", "LLM_PROVIDER": "none"})
!python -m training.benchmark --llm-samples 200 --local-llm Qwen/Qwen2.5-3B-Instruct

## 6. Download the results — send `benchmark_results.zip` to Claude

In [ ]:
!rm -f benchmark_results.zip && zip -q benchmark_results.zip reports/benchmark.json reports/benchmark.md && ls -la benchmark_results.zip
from google.colab import files
files.download("benchmark_results.zip")